# Đánh giá Giả thuyết Mức độ Dàn ý (Conditions A → B → C → D → E) trên Google Colab

Notebook này dùng để kiểm chứng trực tiếp giả thuyết: **Writing Model cần dàn ý đến mức độ nào (`A`, `B`, `C`, `D`, hay `E`)** trên mô hình nền `Qwen2.5-7B-Instruct`.

| Điều kiện | Tên gọi | Cấu trúc dàn ý truyền vào Writing Model |
|---|---|---|
| **A** | Direct Single-Pass | Không có dàn ý (viết thẳng 1 lượt từ premise) |
| **B** | Unstructured Outline | Tóm tắt văn xuôi tự do (200–350 từ), không Event Graph / DAG / Story Arc |
| **C** | Linear Chapter List | Danh sách $K$ chương tuyến tính (`Chapter i + Goal` + chia budget 25/45/30), chưa có Event Graph nhân quả |
| **D** | **Macro Event Graph Only (Đề xuất lõi)** | Đồ thị sự kiện vĩ mô $G=(V,E)$ + Story Arc 5 giai đoạn + chia ngân sách cảnh, **bỏ qua** phân rã micro sub-events |
| **E** | Full 2-Level Hierarchy | Đầy đủ Event Graph $G=(V,E)$ + `SubEventAgent` tách 3 micro-beats/event + DAG sanitizer |

**Thiết kế 3 bước VRAM tách biệt (chạy an toàn trên 1 GPU Colab A100 40GB / L4 24GB)**:
- **Phase 1 (Sinh truyện A–E)**: Nạp `Qwen2.5-7B-Instruct` qua `vLLM` để sinh lần lượt 5 điều kiện A–E (tự động cache từng mẫu ra Google Drive nhờ `--resume`), sau đó giải phóng sạch VRAM.
- **Phase 2b (NLI)**: Nạp `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli` chấm **DOC-NLI Plan Adherence ($S_{\text{adh}}$)**, **Causal Coherence ($S_{\text{causal}}$)**, **Story Arc ($S_{\text{arc}}$)**, rồi giải phóng VRAM.
- **Phase 2c (Local Qwen Judge — không cần OpenAI key)**: Nạp model Qwen lớn hơn (`Qwen2.5-32B-Instruct-AWQ` trên A100 40GB hoặc `Qwen2.5-14B-Instruct-AWQ` trên L4 24GB) để chấm 5 chiều Rubric (`S_coh`, `C_weighted`).

## Bước 1: Kiểm tra GPU Colab & Cài đặt thư viện

In [ ]:
!nvidia-smi

import torch
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
VRAM_GB = 0.0
if torch.cuda.is_available():
    VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", VRAM_GB)
    print("BF16 Supported:", torch.cuda.is_bf16_supported())
else:
    print("Cảnh báo: Chưa bật GPU! Vào Runtime -> Change runtime type -> chọn A100 hoặc L4.")


In [ ]:
# Cài đặt thư viện chạy vLLM + DeBERTa NLI + trực quan hóa bảng/biểu đồ
%pip install -q vllm transformers accelerate sentencepiece protobuf datasets pandas matplotlib


## Bước 2: Mount Google Drive & Nạp mã nguồn `MA-GRPO`

Cell dưới đây tự động nạp mã nguồn theo thứ tự ưu tiên:
1. Nếu có điền `GITHUB_TOKEN`: `git clone` (hoặc `git pull`) bản mới nhất từ GitHub (tránh dùng nhầm file `.zip` cũ trên Drive).
2. Thư mục `MA-GRPO` đã có sẵn bản mới trên Google Drive (`/content/drive/MyDrive/MA-GRPO`).
3. File nén `/content/drive/MyDrive/MA-GRPO.zip` (tự kiểm tra có chứa `scripts/run_macro_planning_benchmark.py` trước khi giải nén).

In [ ]:
import os
import shutil
import subprocess
import zipfile

# 1. Mount Google Drive để lưu bền vững cache & kết quả A-E
USE_DRIVE = True
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = "/content/drive/MyDrive/MA-GRPO-Ablation-ABCDE"
    except Exception as e:
        print("Không mount được Drive, dùng thư mục cục bộ:", e)
        OUTPUT_DIR = "./benchmark_outputs"
else:
    OUTPUT_DIR = "./benchmark_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Thư mục lưu kết quả:", OUTPUT_DIR)

# 2. Nạp mã nguồn MA-GRPO
REPO_DIR = "/content/MA-GRPO"
DRIVE_REPO_DIR = "/content/drive/MyDrive/MA-GRPO"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/MA-GRPO.zip"
REPO_URL = "https://github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git"
GITHUB_TOKEN = ""  # Dán GitHub PAT nếu repo private (có token sẽ ưu tiên clone bản mới từ GitHub)

def _has_benchmark_script(path: str) -> bool:
    return os.path.exists(os.path.join(path, "scripts", "run_macro_planning_benchmark.py"))

if _has_benchmark_script(os.getcwd()) and not (GITHUB_TOKEN.strip() and os.path.abspath(os.getcwd()) == "/content"):
    REPO_DIR = os.getcwd()
    print("Đang ở sẵn trong thư mục repo:", REPO_DIR)
else:
    if os.path.abspath(os.getcwd()).startswith(os.path.abspath(REPO_DIR)):
        os.chdir("/content")

    token = GITHUB_TOKEN.strip()
    if token:
        if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, ".git")):
            print(f"Xóa thư mục {REPO_DIR} cũ để clone lại từ GitHub...")
            shutil.rmtree(REPO_DIR, ignore_errors=True)
        if os.path.exists(os.path.join(REPO_DIR, ".git")):
            print(f"Cập nhật repo tại {REPO_DIR} (git pull)...")
            subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
        else:
            clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
            print(f"Cloning từ GitHub vào {REPO_DIR}...")
            res = subprocess.run(["git", "clone", clone_url, REPO_DIR])
            if res.returncode != 0:
                raise RuntimeError("Không clone được repo từ GitHub. Hãy kiểm tra lại GITHUB_TOKEN và quyền truy cập repo.")
    elif not _has_benchmark_script(REPO_DIR):
        if _has_benchmark_script(DRIVE_REPO_DIR):
            print(f"Copying repo từ Google Drive ({DRIVE_REPO_DIR}) sang {REPO_DIR}...")
            shutil.copytree(DRIVE_REPO_DIR, REPO_DIR, dirs_exist_ok=True)
        elif os.path.exists(DRIVE_ZIP_PATH):
            print(f"Kiểm tra & giải nén {DRIVE_ZIP_PATH} sang /content...")
            with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
                has_new_script = any(name.endswith("scripts/run_macro_planning_benchmark.py") for name in zf.namelist())
                if not has_new_script:
                    raise RuntimeError(
                        f"File {DRIVE_ZIP_PATH} trên Google Drive là bản CŨ (thiếu scripts/run_macro_planning_benchmark.py). "
                        "Hãy điền GITHUB_TOKEN ở trên để clone từ GitHub, hoặc xóa/cập nhật lại MA-GRPO.zip trên Drive."
                    )
                zf.extractall("/content")
            if not _has_benchmark_script(REPO_DIR):
                for root, dirs, files in os.walk("/content"):
                    dirs[:] = [d for d in dirs if d != "drive"]
                    if "run_macro_planning_benchmark.py" in files and os.path.basename(root) == "scripts":
                        REPO_DIR = os.path.dirname(root)
                        break
        else:
            print(f"Cloning từ GitHub vào {REPO_DIR}...")
            res = subprocess.run(["git", "clone", REPO_URL, REPO_DIR])
            if res.returncode != 0:
                raise RuntimeError(
                    "Không clone được repo từ GitHub (repo private cần điền GITHUB_TOKEN ở trên, "
                    "hoặc upload bản MA-GRPO mới lên Google Drive)."
                )

os.chdir(REPO_DIR)
assert os.path.exists("scripts/run_macro_planning_benchmark.py"), (
    f"Không tìm thấy scripts/run_macro_planning_benchmark.py trong {os.getcwd()}!"
)
print("Working directory:", os.getcwd())
print("Datasets trong data/:", sorted(f for f in os.listdir("data") if f.endswith(".jsonl")))


## Bước 3: Kiểm tra Unit Test (Sanity Check trước khi chạy GPU)

In [ ]:
!python -m unittest discover -s tests -v


## Bước 4: Cấu hình thí nghiệm Track 2 (A → B → C → D → E) & Local Qwen Judge

- **Bước 5 (bên dưới) là Track 2 (chấm A, B, C, D, E viết truyện dài)**; **Bước 6 là Track 1 (tùy chọn, VR-CLI trên `longstory_test.jsonl`)**.
- **`NUM_PREMISES_TRACK2`**: Số đề bài chạy cho mỗi điều kiện trong 5 điều kiện `A, B, C, D, E` (ví dụ `5` premise × 5 điều kiện = 25 truyện). Có `--resume` lưu Google Drive nên có thể chạy pilot `3–5` mẫu trước rồi tăng lên `15–30` mẫu.
- **Local Judge (không cần OpenAI API key)**:
  - Trên **A100 40GB**: mặc định `Qwen/Qwen2.5-32B-Instruct-AWQ` (4-bit ~19 GB VRAM, chừa ~15 GB KV-cache với `vLLM`; bản 32B BF16 tốn ~64 GB nên sẽ OOM trên A100 40GB).
  - Trên **L4 24GB**: mặc định `Qwen/Qwen2.5-14B-Instruct-AWQ`.
  - Trên **T4 16GB**: tự tắt Judge để tránh OOM (chỉ chấm DeBERTa NLI + các chỉ số độ dài/từ vựng).

In [ ]:
import os
import subprocess
import torch

def _run_streaming(cmd):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
NLI_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

# Dataset cho so sánh A-B-C-D-E (Sinh truyện dài)
TRACK2_DATA = "data/tell-me-a-story-test.jsonl"   # Hoặc "data/mops_premises.jsonl" / "data/writingprompts_test.jsonl"
# Dataset cho Bước 6 (Tùy chọn): Track 1 VR-CLI Next-Chapter Prediction (B-C-D-E)
TRACK1_DATA = "data/longstory_test.jsonl"          # Hoặc "data/booksum_chapters.jsonl"

# Quy mô thử nghiệm A-E
CONDITIONS = ["A", "B", "C", "D", "E"]
NUM_PREMISES_TRACK2 = 5     # Pilot nhanh: 3-5 | Chạy paper: 15-30 (có --resume nên nâng dần thoải mái)
TARGET_WORDS = 8000         # Hoặc 10000
NUM_CHAPTERS = 8

# Tự động chọn cấu hình Generator & Local Qwen Judge theo loại GPU Colab
vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9) if torch.cuda.is_available() else 0.0
if vram_gb >= 35:
    BACKEND = "vllm"
    GPU_MEM_UTIL = 0.80
    MAX_MODEL_LEN = 32768   # Mở kịch kim context gốc 32K của Qwen2.5-7B-Instruct (không bị cắt max_tokens)
    ENABLE_JUDGE = True
    JUDGE_BACKEND = "local_vllm"
    # Mặc định dùng bản 32B AWQ 4-bit (~19GB VRAM, chừa ~15GB KV-cache) vì BF16 32B (~64GB) sẽ OOM trên A100 40GB
    JUDGE_MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"
    JUDGE_GPU_MEM_UTIL = 0.85
    JUDGE_MAX_MODEL_LEN = 16384
elif vram_gb >= 20:
    BACKEND = "vllm"
    GPU_MEM_UTIL = 0.85
    MAX_MODEL_LEN = 16384
    ENABLE_JUDGE = True
    JUDGE_BACKEND = "local_vllm"
    JUDGE_MODEL = "Qwen/Qwen2.5-14B-Instruct-AWQ"
    JUDGE_GPU_MEM_UTIL = 0.88
    JUDGE_MAX_MODEL_LEN = 12288
else:
    BACKEND = "hf"          # An toàn hơn trên T4 16GB tránh OOM KV-cache vLLM
    GPU_MEM_UTIL = 0.65
    MAX_MODEL_LEN = 8192
    ENABLE_JUDGE = False
    JUDGE_BACKEND = "auto"
    JUDGE_MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"
    JUDGE_GPU_MEM_UTIL = 0.80
    JUDGE_MAX_MODEL_LEN = 8192

print(f"GPU VRAM: {vram_gb:.1f} GB -> Backend: {BACKEND} | max_model_len: {MAX_MODEL_LEN} | Conditions: {CONDITIONS}")
print(f"Judge Enabled: {ENABLE_JUDGE} | Judge Backend: {JUDGE_BACKEND} | Judge Model: {JUDGE_MODEL} (max_len={JUDGE_MAX_MODEL_LEN})")
print(f"Track 2 Data: {TRACK2_DATA} (exists={os.path.exists(TRACK2_DATA)}) | Samples: {NUM_PREMISES_TRACK2} | Target: {TARGET_WORDS} words / {NUM_CHAPTERS} ch")


## Bước 5 (Chính — TRACK 2): Chạy Ablation 5 mức độ dàn ý A, B, C, D, E

Lưu ý: **Bước này chính là Track 2 (sinh truyện dài cho A–E)**, không phải Track 1.
- Sinh truyện dài với 5 điều kiện `A`, `B`, `C`, `D`, `E` trên `TRACK2_DATA`.
- Tự giải phóng Qwen-7B → chấm DeBERTa-v3 NLI (`S_adh`, `S_causal`, `S_prog`, `S_arc`) → tự giải phóng NLI → nạp Local Qwen Judge (`32B-AWQ` trên A100 / `14B-AWQ` trên L4) chấm 5 chiều (`S_coh`, `C_weighted`) nếu `ENABLE_JUDGE = True`.

In [ ]:
cmd_track2 = [
    "python", "-u", "scripts/run_macro_planning_benchmark.py",
    "--mode", "track2",
    "--backend", BACKEND,
    "--model", MODEL_ID,
    "--nli-model", NLI_MODEL,
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--track2-data", TRACK2_DATA,
    "--track2-samples", str(NUM_PREMISES_TRACK2),
    "--target-words", str(TARGET_WORDS),
    "--num-chapters", str(NUM_CHAPTERS),
    "--conditions", *CONDITIONS,
    "--output-dir", OUTPUT_DIR,
    "--resume",
]
if ENABLE_JUDGE:
    cmd_track2.extend([
        "--enable-judge",
        "--judge-backend", JUDGE_BACKEND,
        "--judge-model", JUDGE_MODEL,
        "--judge-gpu-memory-utilization", str(JUDGE_GPU_MEM_UTIL),
        "--judge-max-model-len", str(JUDGE_MAX_MODEL_LEN),
    ])

print("Running command:\n", " ".join(cmd_track2), flush=True)
_run_streaming(cmd_track2)


## Bước 6 (Tùy chọn — Chạy rất nhanh): Track 1 VR-CLI Next-Chapter Prediction (`B` vs `C` vs `D` vs `E`)

Ngoài hiệu quả viết truyện ở Bước 5, bạn có thể chạy thêm cell này để đo **năng lực dự đoán chương sách thật $y_{\text{gold}}$** trên `LONGSTORY` (không tốn công sinh truyện dài, chỉ sinh dàn ý và đo độ giảm Perplexity $I_\pi$ giữa B, C, D, E).

In [ ]:
RUN_TRACK1_VRCLI = False  # Đổi thành True nếu muốn đo thêm I_pi (VR-CLI PPL Reduction) cho B, C, D, E
TRACK1_SAMPLES = 20

if RUN_TRACK1_VRCLI and os.path.exists(TRACK1_DATA):
    cmd_track1 = [
        "python", "-u", "scripts/run_macro_planning_benchmark.py",
        "--mode", "track1",
        "--backend", BACKEND,
        "--model", MODEL_ID,
        "--nli-model", NLI_MODEL,
        "--gpu-memory-utilization", str(GPU_MEM_UTIL),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--track1-data", TRACK1_DATA,
        "--track1-samples", str(TRACK1_SAMPLES),
        "--conditions", "B", "C", "D", "E",
        "--output-dir", OUTPUT_DIR,
        "--resume",
    ]
    print("Running Track 1:\n", " ".join(cmd_track1), flush=True)
    _run_streaming(cmd_track1)
else:
    print("Bỏ qua Track 1 (đặt RUN_TRACK1_VRCLI = True nếu muốn chạy thêm VR-CLI trên LONGSTORY).")


## Bước 7: Hiển thị Bảng So sánh A → B → C → D → E, Biểu đồ & Kết luận Giả thuyết H1–H4

In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

t2_csv = os.path.join(OUTPUT_DIR, "track2_ablation_5conditions.csv")
t1_csv = os.path.join(OUTPUT_DIR, "track1_vrcli_results.csv")
summary_json = os.path.join(OUTPUT_DIR, "benchmark_summary.json")

if os.path.exists(t2_csv):
    df2 = pd.read_csv(t2_csv)
    key_cols = [
        c for c in [
            "condition", "n_samples", "planning_tokens", "actual_words", "target_words",
            "length_score_sl", "early_term_rate_pct", "S_adh", "S_causal", "S_arc",
            "Distinct-3 (%)", "burstiness_std", "generation_latency_s", "S_coh", "C_weighted"
        ] if c in df2.columns
    ]
    print("=== BẢNG SO SÁNH 5 MỨC ĐỘ DÀN Ý (A -> B -> C -> D -> E) ===")
    display(df2[key_cols])

    # Vẽ 4 biểu đồ so sánh A-E
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    conds = df2["condition"].astype(str).tolist()
    colors = ["#9aa0a6", "#fbbc04", "#4285f4", "#34a853", "#ea4335"][:len(conds)]

    # 1. Độ dài thực tế vs Target Words
    axes[0, 0].bar(conds, df2["actual_words"], color=colors)
    if "target_words" in df2.columns and len(df2) > 0:
        axes[0, 0].axhline(df2["target_words"].iloc[0], color="black", linestyle="--", label="Target Words")
        axes[0, 0].legend()
    axes[0, 0].set_title("Độ dài thực tế (Actual Words)")
    axes[0, 0].set_ylabel("Words")

    # 2. Plan Adherence (DOC-NLI), Causal & Story Arc
    x = range(len(conds))
    w = 0.25
    axes[0, 1].bar([i - w for i in x], df2["S_adh"].fillna(0), width=w, label="S_adh (DOC-NLI)")
    axes[0, 1].bar(list(x), df2["S_causal"].fillna(0), width=w, label="S_causal")
    axes[0, 1].bar([i + w for i in x], df2["S_arc"].fillna(0), width=w, label="S_arc (Freytag)")
    axes[0, 1].set_xticks(list(x))
    axes[0, 1].set_xticklabels(conds)
    axes[0, 1].set_ylim(0, 1.05)
    axes[0, 1].set_title("Chất lượng Kế hoạch & Độ bám Dàn ý")
    axes[0, 1].legend()

    # 3. Planning Tokens (So sánh chi phí lập kế hoạch D vs E)
    axes[1, 0].bar(conds, df2["planning_tokens"].fillna(0), color=colors)
    axes[1, 0].set_title("Chi phí Planning Tokens (thấp hơn = gọn hơn)")
    axes[1, 0].set_ylabel("Tokens")

    # 4. Distinct-3 (%)
    axes[1, 1].bar(conds, df2["Distinct-3 (%)"].fillna(0), color=colors)
    axes[1, 1].set_title("Độ đa dạng từ vựng Distinct-3 (%)")
    axes[1, 1].set_ylabel("%")

    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, "ablation_abcde_comparison.png")
    plt.savefig(chart_path, dpi=150)
    plt.show()
    print("Đã lưu biểu đồ tại:", chart_path)

if os.path.exists(t1_csv):
    print("\n=== TRACK 1 (VR-CLI NEXT-CHAPTER PREDICTION) ===")
    display(pd.read_csv(t1_csv))

if os.path.exists(summary_json):
    with open(summary_json, "r", encoding="utf-8") as f:
        summary = json.load(f)
    audit = summary.get("hypotheses_audit", {})
    if audit:
        print("\n=== KIỂM ĐỊNH GIẢ THUYẾT TỐI THIỂU HÓA DÀN Ý (H1–H4) ===")
        print(json.dumps(audit, indent=2, ensure_ascii=False))


## Bước 8 (Chạy qua đêm): Đồng bộ Drive & Tự động ngắt Server Colab (Tiết kiệm Compute Units)

Khi bạn chạy nối tiếp bằng **Run after (`Ctrl+F10` từ Bước 4 hoặc Bước 5)** rồi tắt máy đi ngủ, cell cuối này sẽ chờ đồng bộ toàn bộ file CSV/JSON/PNG lên Google Drive rồi gọi `runtime.unassign()` để **thu hồi GPU A100 ngay lập tức**, tránh tiêu tốn đơn vị điện toán suốt đêm.

In [ ]:
import time

AUTO_DISCONNECT_RUNTIME = True  # Đổi False nếu bạn đang ngồi xem trực tiếp và muốn giữ máy ảo GPU

if AUTO_DISCONNECT_RUNTIME:
    try:
        from google.colab import drive, runtime
        print("Đang chờ đồng bộ toàn bộ file kết quả lên Google Drive (10s)...", flush=True)
        time.sleep(10)
        try:
            drive.flush_and_unmount()
        except Exception as e:
            print("Cảnh báo khi flush Drive:", e, flush=True)
        print("Đã đồng bộ xong. Tự động thu hồi server Colab (runtime.unassign) trong 5 giây...", flush=True)
        time.sleep(5)
        runtime.unassign()
    except ImportError:
        print("Không chạy trong Google Colab -> bỏ qua bước tự ngắt runtime.")
else:
    print("AUTO_DISCONNECT_RUNTIME = False -> giữ nguyên Runtime GPU.")
